# Mobilidade Urbana (app 3) — visual no padrão dos widgets dos DEVs — v12
Troca os sinais de "app feito por IA" pelo padrão Calcite usado em `engemap/arcgis-exb-widgets`:
- **Horários da Linha v8**: painel branco, botões `CalciteButton`, ícones Calcite (sem emojis), interruptor `CalciteSwitch`,
  dicas em `CalciteNotice`, sem selos coloridos; destaque da próxima saída no dourado dos DEVs (#F5C400).
- **Novo widget Serviços ao Passageiro** no lugar dos 3 cartões de texto + 4 botões pílula.
- **Lista**: ícone de ônibus do Calcite em cinza, sem círculo; título alinhado à esquerda; seleção discreta.
- A casca do CCTV (cabeçalho #4463a9, logo, busca, ferramentas) **não muda**. Backups em `Documents\Backup_ExB`.

⚠ Depois: **reinicie o ExB** (Iniciar_ExB.bat — widget novo) → construtor **Ctrl+F5 → mudança pequena → Salvar → Publicar** → teste com `?draft=true` ou na prévia.
Saída: pasta **Saidas** do projeto aberto no Pro → `Mobilidade_Visual_DEVs_v12.txt`

In [ ]:
# ===== CÉLULA 1 — Configuração =====
import os, json, datetime, shutil
NB_NOME = "Mobilidade_Visual_DEVs_v12"
EXB = r"C:\ExB_1.19"
CFG_APP = os.path.join(EXB, "server", "public", "apps", "3", "resources", "config", "config.json")
WIDGETS_DIR = os.path.join(EXB, "client", "your-extensions", "widgets")
try:
    import arcpy
    PASTA_SAIDA = os.path.join(arcpy.mp.ArcGISProject("CURRENT").homeFolder, "Saidas")
except Exception:
    PASTA_SAIDA = os.path.join(os.path.expanduser("~"), "Documents", "Saidas_Notebooks")
os.makedirs(PASTA_SAIDA, exist_ok=True)
PASTA_BKP = os.path.join(os.path.expanduser("~"), "Documents", "Backup_ExB", f"mobilidade_v12_{datetime.datetime.now():%Y%m%d_%H%M%S}")
os.makedirs(PASTA_BKP, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f: f.write(s + "\n")
log("▶ CÉLULA 1 — Configuração: iniciando...")
for p in (CFG_APP, WIDGETS_DIR):
    if not os.path.exists(p): raise RuntimeError("Não encontrado: " + p)
log("  Saída (.txt):", ARQ_TXT)
log("  Backups:", PASTA_BKP)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Widgets próprios (Horários da Linha v8 + Serviços ao Passageiro) =====
log("\n▶ CÉLULA 2 — Widgets: iniciando...")
WIDGETS = {
  'horarios-linha': {
    'manifest.json': r'''{
  "name": "horarios-linha",
  "label": "Horários da Linha",
  "type": "widget",
  "version": "1.19.0",
  "exbVersion": "1.19.0",
  "author": "GeoAPUC",
  "description": "Mostra a tabela de horários (dias úteis) da linha de ônibus selecionada na lista, com a próxima saída em destaque.",
  "copyright": "",
  "license": "",
  "properties": {},
  "translatedLocales": [
    "en"
  ],
  "defaultSize": {
    "width": 380,
    "height": 560
  },
  "dependency": [
    "jimu-arcgis"
  ]
}''',
    'config.json': r'''{
  "dataSourceId": "dataSource_1-linhas_onibus",
  "linhasUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0",
  "horariosUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Horarios_Onibus/FeatureServer/0",
  "mobilibusUrl": "https://editor.mobilibus.com/web/detalhes-linha/4abp9",
  "mobilibusLinhas": {
    "110": "fpt",
    "115": "2cjn",
    "120": "fqg",
    "125": "fqa",
    "130": "fqp",
    "135": "fq7",
    "140": "fq3",
    "145": "fq4",
    "150": "fpo",
    "155": "fr0",
    "160": "fqt",
    "165": "fqo",
    "170": "fr1",
    "175": "6ckh",
    "195": "fpg",
    "200": "fpm",
    "205": "fps",
    "210": "fqr",
    "215": "hrte",
    "220": "fpr",
    "230": "fq8",
    "235": "fqk",
    "240": "fph",
    "270": "9t55",
    "320": "fpu",
    "400": "fqn",
    "500": "fq6",
    "998": "fqd",
    "999": "fqb"
  },
  "paradasUrl": "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Paradas_Onibus/FeatureServer/0",
  "paradasLayerId": "paradas_onibus",
  "paradasLigadoAoAbrir": false
}''',
    'icon.svg': r'''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 16 16" width="16" height="16"><rect x="3" y="1.5" width="10" height="11" rx="2" fill="none" stroke="#000" stroke-width="1.2"/><rect x="4.6" y="3.2" width="6.8" height="3.6" rx=".6" fill="#000"/><circle cx="5.5" cy="9.6" r=".9" fill="#000"/><circle cx="10.5" cy="9.6" r=".9" fill="#000"/><rect x="4" y="12.5" width="2" height="2" rx=".5" fill="#000"/><rect x="10" y="12.5" width="2" height="2" rx=".5" fill="#000"/></svg>
''',
    'src/runtime/widget.tsx': r'''// Horários da Linha – GeoAPUC (v8: visual no padrão dos widgets dos DEVs — Calcite, sem emojis/pílulas/selos)
// Escrito sem JSX (React.createElement) para não depender da configuração de compilação.
import { React, getAppStore, type AllWidgetProps } from 'jimu-core'
import { JimuMapViewComponent } from 'jimu-arcgis'
import { CalciteButton, CalciteIcon, CalciteNotice, CalciteSwitch } from 'calcite-components'

const h = React.createElement
const { useState, useEffect } = React

// paleta dos widgets dos DEVs (hub) + azul da casca homologada
const C = {
  fundo: '#FFFFFF', fundo2: '#F5F7FA', borda: '#E5E7EB', texto: '#1F2937', texto2: '#4B5563', suave: '#6B7280',
  apagado: '#9CA3AF', marca: '#4463A9', ouro: '#F5C400'
}
const FONTE = "Inter, 'Avenir Next', Roboto, Arial, sans-serif"

function hhmm (v: any): string {
  const s = String(v ?? '').trim()
  const m = s.match(/^(\d{1,2}):(\d{2})/)
  return m ? `${m[1].padStart(2, '0')}:${m[2]}` : s
}
function minutos (v: string): number {
  const m = v.match(/^(\d{2}):(\d{2})/)
  return m ? Number(m[1]) * 60 + Number(m[2]) : -1
}
function chaveSentido (v: any): string {
  return String(v ?? '').normalize('NFD').replace(/[̀-ͯ]/g, '').toLowerCase()
    .replace(/sentido para/g, '').replace(/[^a-z0-9]/g, '')
}
function aspas (v: any): string { return String(v ?? '').replace(/'/g, "''") }

async function consultar (url: string, where: string, campos: string): Promise<any[]> {
  const q = `${url}/query?where=${encodeURIComponent(where)}&outFields=${campos}&returnGeometry=false&f=json`
  const j: any = await fetch(q).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro na consulta')
  return (j.features || []).map((f: any) => f.attributes || {})
}
async function contar (url: string, where: string): Promise<number> {
  const q = `${url}/query?where=${encodeURIComponent(where)}&returnCountOnly=true&f=json`
  const j: any = await fetch(q).then(r => r.json())
  if (j.error) throw new Error(j.error.message || 'erro na contagem')
  return Number(j.count || 0)
}

export default function Widget (props: AllWidgetProps<any>) {
  const cfg: any = props.config || {}
  const dsId: string = cfg.dataSourceId || 'dataSource_1-linhas_onibus'
  const urlLin: string = cfg.linhasUrl
  const urlHor: string = cfg.horariosUrl
  const urlPar: string = cfg.paradasUrl
  const idCamadaParadas: string = cfg.paradasLayerId || 'paradas_onibus'
  const mbBase: string = cfg.mobilibusUrl || 'https://editor.mobilibus.com/web/detalhes-linha/4abp9'
  const mbCod: any = cfg.mobilibusLinhas || {}
  const linkTempoReal = (cod: any): string => {
    const c = mbCod[String(cod ?? '').trim()]
    return c ? `${mbBase}#${c}` : mbBase
  }

  // ---- linha selecionada na lista (estado do app) ----
  const lerSelecao = (): string | null => {
    try {
      const inf: any = (getAppStore().getState() as any)?.dataSourcesInfo || {}
      for (const k of Object.keys(inf)) {
        if (k === dsId || k.startsWith(dsId + '-')) {
          const ids = inf[k]?.selectedIds
          if (ids && ids.length) return String(ids[0])
        }
      }
    } catch (e) { console.warn('[Horários da Linha] seleção', e) }
    return null
  }
  const [oid, setOid] = useState<string | null>(lerSelecao())
  useEffect(() => {
    const store: any = getAppStore()
    const desfazer = store.subscribe(() => {
      const novo = lerSelecao()
      setOid(atual => (atual === novo ? atual : novo))
    })
    return () => { try { desfazer() } catch (e) {} }
  }, [dsId])

  const [linha, setLinha] = useState<any>(null)
  const [hor, setHor] = useState<any[]>([])
  const [planoB, setPlanoB] = useState<boolean>(false)
  const [carregando, setCarregando] = useState<boolean>(false)
  const [erro, setErro] = useState<string | null>(null)
  const [agora, setAgora] = useState<Date>(new Date())
  useEffect(() => { const t = setInterval(() => setAgora(new Date()), 60000); return () => clearInterval(t) }, [])

  // ---- paradas (camada Mobilibus, filtro pelo campo 'viagens') ----
  const [jmv, setJmv] = useState<any>(null)
  const [mostrar, setMostrar] = useState<boolean>(!!cfg.paradasLigadoAoAbrir)
  const [nParadas, setNParadas] = useState<number | null>(null)
  useEffect(() => {
    const v: any = jmv?.view
    if (!v || !urlPar) return
    const lyr: any = v.map.allLayers.find((l: any) => l.id === idCamadaParadas)
    if (!lyr) { console.warn('[Horários da Linha] camada de paradas não encontrada:', idCamadaParadas); return }
    let vivo = true
    if (!mostrar) { lyr.visible = false; setNParadas(null); return }
    ;(async () => {
      try {
        if (!oid) { lyr.definitionExpression = '1=0'; lyr.visible = true; if (vivo) setNParadas(0); return }
        const l = (await consultar(urlLin, `objectid=${Number(oid)}`, 'cod_linha,sentido'))[0] || {}
        const cod = String(l.cod_linha ?? '').trim()
        let where = `viagens LIKE '%;${aspas(cod + ':' + chaveSentido(l.sentido))};%'`
        let n = await contar(urlPar, where)
        if (!n) { where = `linhas LIKE '%,${aspas(cod)},%'`; n = await contar(urlPar, where) }
        if (!vivo) return
        lyr.definitionExpression = n ? where : '1=0'
        lyr.visible = true
        setNParadas(n)
      } catch (e) {
        console.warn('[Horários da Linha] paradas', e)
        if (vivo) setNParadas(null)
      }
    })()
    return () => { vivo = false }
  }, [jmv, mostrar, oid, urlPar, urlLin, idCamadaParadas])

  // nome da parada ao passar o mouse (ou tocar)
  useEffect(() => {
    const v: any = jmv?.view
    if (!v) return
    const lyr: any = v.map.allLayers.find((l: any) => l.id === idCamadaParadas)
    if (!lyr) return
    try { lyr.outFields = ['nome'] } catch (e) {}
    const dica = document.createElement('div')
    Object.assign(dica.style, { position: 'absolute', display: 'none', pointerEvents: 'none', zIndex: '99', maxWidth: '260px',
      background: '#FFFFFF', color: C.texto, border: `1px solid ${C.borda}`, borderRadius: '4px', padding: '4px 8px',
      fontSize: '12px', fontWeight: '500', fontFamily: FONTE, boxShadow: '0 2px 4px rgba(0,0,0,0.1)', whiteSpace: 'nowrap' })
    v.container.appendChild(dica)
    let ocupado = false
    const mostrarDica = (ev: any) => {
      if (ocupado || !lyr.visible) { if (!lyr.visible) dica.style.display = 'none'; return }
      ocupado = true
      v.hitTest(ev, { include: [lyr] }).then((res: any) => {
        ocupado = false
        const g = (res?.results || []).find((r: any) => r.graphic?.layer?.id === idCamadaParadas)?.graphic
        if (g) {
          dica.textContent = 'Parada: ' + (g.attributes?.nome || 'sem nome')
          dica.style.left = (ev.x + 14) + 'px'; dica.style.top = (ev.y + 14) + 'px'; dica.style.display = 'block'
          v.container.style.cursor = 'pointer'
        } else { dica.style.display = 'none'; v.container.style.cursor = '' }
      }).catch(() => { ocupado = false })
    }
    const h1 = v.on('pointer-move', mostrarDica)
    const h2 = v.on('click', mostrarDica)
    const h3 = v.on('pointer-leave', () => { dica.style.display = 'none' })
    return () => { try { h1.remove(); h2.remove(); h3.remove(); dica.remove() } catch (e) {} }
  }, [jmv, idCamadaParadas])

  // ---- horários da linha ----
  useEffect(() => {
    let vivo = true
    if (!oid) { setLinha(null); setHor([]); return }
    setCarregando(true); setErro(null)
    ;(async () => {
      try {
        const l = (await consultar(urlLin, `objectid=${Number(oid)}`, 'objectid,id_rota,cod_linha,nome,sentido'))[0]
        if (!l) throw new Error('linha não encontrada')
        let rows = await consultar(urlHor, `id_rota='${aspas(l.id_rota)}'`, 'id_rota,ponto,horario,sentido,observacao')
        let b = false
        if (!rows.length) {
          const cod = String(l.id_rota || '').split('_')[0]
          rows = (await consultar(urlHor, `id_rota LIKE '${aspas(cod)}%'`, 'id_rota,ponto,horario,sentido,observacao'))
            .filter((r: any) => String(r.id_rota || '').startsWith(cod + '_'))
          b = rows.length > 0
        }
        rows.forEach((r: any) => { r._h = hhmm(r.horario); r._m = minutos(r._h) })
        rows.sort((a: any, z: any) => a._m - z._m)
        if (vivo) { setLinha(l); setHor(rows); setPlanoB(b); setCarregando(false) }
      } catch (e: any) {
        console.warn('[Horários da Linha]', e)
        if (vivo) { setErro('Não foi possível carregar os horários.'); setCarregando(false) }
      }
    })()
    return () => { vivo = false }
  }, [oid, urlLin, urlHor])

  // ---- peças visuais ----
  const mapaId: any = (props as any).useMapWidgetIds?.[0]
  const conector = mapaId ? h(JimuMapViewComponent as any, { useMapWidgetId: mapaId, onActiveViewChange: (v: any) => setJmv(v) }) : null
  const icone = (nome: string, cor: string = C.suave, escala: string = 's') =>
    h(CalciteIcon as any, { icon: nome, scale: escala, style: { color: cor, flex: 'none' } })
  const rotulo = (txt: string) => h('div', { style: { fontSize: 11, fontWeight: 500, letterSpacing: 0.3, color: C.suave, textTransform: 'uppercase' } }, txt)
  const aviso = (kind: string, ic: string, titulo: string | null, msg: any, link: any = null) =>
    h(CalciteNotice as any, { open: true, kind, icon: ic, scale: 's', width: 'full' },
      titulo ? h('div', { slot: 'title' }, titulo) : null,
      h('div', { slot: 'message' }, msg),
      link ? h('div', { slot: 'link' }, link) : null)

  const cabecalho = h('div', { style: { display: 'flex', alignItems: 'center', justifyContent: 'space-between', gap: 8 } },
    h('div', { style: { fontSize: 15, fontWeight: 600, color: C.texto } }, 'Tabela de horários'),
    h('label', { style: { display: 'flex', alignItems: 'center', gap: 6, fontSize: 12, color: C.texto2, cursor: 'pointer', userSelect: 'none' },
                 title: 'Mostrar no mapa só as paradas da linha selecionada' },
      h('span', null, 'Paradas' + (mostrar && nParadas !== null ? ` (${nParadas})` : '')),
      h(CalciteSwitch as any, { scale: 's', checked: mostrar, onCalciteSwitchChange: (e: any) => setMostrar(!!e?.target?.checked) })))

  const caixa = (filhos: any[]) => h('div', {
    className: 'jimu-widget horarios-linha',
    style: { width: '100%', height: '100%', boxSizing: 'border-box', background: C.fundo, color: C.texto, borderRadius: 8,
             border: `1px solid ${C.borda}`, padding: 14, overflowY: 'auto', fontFamily: FONTE,
             display: 'flex', flexDirection: 'column', gap: 12 }
  }, conector, cabecalho, ...filhos)

  const botaoTempoReal = (cod: any, appearance: string = 'solid') => h(CalciteButton as any, {
    href: linkTempoReal(cod), target: '_blank', rel: 'noopener noreferrer', kind: 'brand', appearance, scale: 'm', width: 'full',
    iconStart: 'bus', iconEnd: 'launch'
  }, cod ? `Ônibus da linha ${cod} em tempo real` : 'Ônibus em tempo real')

  const ligarParadas = h(CalciteButton as any, { appearance: 'transparent', kind: 'brand', scale: 's', iconStart: 'map-pin',
    onClick: () => setMostrar(true) }, 'Mostrar paradas')

  if (!oid) {
    return caixa([
      h('div', { style: { display: 'flex', flexDirection: 'column', alignItems: 'center', gap: 8, padding: '24px 8px', textAlign: 'center' } },
        icone('bus', C.apagado, 'l'),
        h('div', { style: { fontSize: 14, color: C.texto2 } }, 'Selecione uma linha na lista para ver os horários.')),
      botaoTempoReal(null, 'outline'),
      aviso('brand', 'lightbulb', null, 'Depois de escolher a linha, ligue “Paradas” para ver no mapa os pontos de ônibus dela.')
    ])
  }
  if (carregando) return caixa([h('div', { style: { margin: 'auto', color: C.suave, fontSize: 13 } }, 'Carregando horários…')])
  if (erro || !linha) return caixa([aviso('danger', 'exclamation-mark-triangle', null, erro || 'Linha não encontrada.')])

  try {
    const dow = agora.getDay()
    const diaUtil = dow >= 1 && dow <= 5
    const minAgora = agora.getHours() * 60 + agora.getMinutes()
    const proxima = diaUtil ? hor.find((r: any) => r._m >= minAgora) : null
    const destaque: string = proxima ? proxima._h : (hor[0]?._h || '—')
    let sub = ''
    if (!hor.length) sub = 'Sem horários na base da Prefeitura'
    else if (!diaUtil) sub = 'Fim de semana · primeira saída em dia útil'
    else if (proxima) { const d = proxima._m - minAgora; sub = d === 0 ? 'Saindo agora' : `Próxima saída · em ${d >= 60 ? Math.floor(d / 60) + ' h ' : ''}${d % 60} min` }
    else sub = 'Sem mais saídas hoje · primeira amanhã'

    const grupos: any = {}
    hor.forEach((r: any) => {
      const k = (planoB ? (r.sentido || 'Outro sentido') + ' · ' : '') + (r.ponto || '—')
      ;(grupos[k] = grupos[k] || []).push(r)
    })
    const obs = Array.from(new Set(hor.map((r: any) => String(r.observacao || '').trim()).filter((x: string) => x && x !== '-')))

    return caixa([
      // linha
      h('div', { style: { borderBottom: `1px solid ${C.borda}`, paddingBottom: 10 } },
        h('div', { style: { display: 'flex', alignItems: 'baseline', gap: 8 } },
          h('span', { style: { fontSize: 20, fontWeight: 700, color: C.marca } }, String(linha.cod_linha ?? '')),
          h('span', { style: { fontSize: 16, fontWeight: 600, color: C.texto, lineHeight: 1.25 } }, linha.nome || '')),
        h('div', { style: { fontSize: 12, color: C.suave, marginTop: 2 } }, `Sentido ${linha.sentido || '—'} · dias úteis`)),
      botaoTempoReal(linha.cod_linha),
      !mostrar ? aviso('brand', 'lightbulb', null,
        `Veja no mapa as paradas da linha ${linha.cod_linha} neste sentido. Passe o mouse sobre uma parada para ver o nome.`, ligarParadas) : null,
      // próxima saída
      h('div', { style: { display: 'flex', alignItems: 'center', gap: 12, background: C.fundo2, borderRadius: 8,
                          borderLeft: `3px solid ${C.ouro}`, padding: '10px 12px' } },
        icone('clock', C.marca, 'm'),
        h('div', null, rotulo(sub || 'Horário'),
          h('div', { style: { fontSize: 28, fontWeight: 700, lineHeight: 1.1, color: C.texto, fontVariantNumeric: 'tabular-nums' } }, destaque))),
      planoB ? aviso('warning', 'information', null,
        `Este sentido não tem horários próprios. Abaixo, as saídas da linha ${linha.cod_linha} nos outros sentidos.`) : null,
      // todos os horários
      ...Object.keys(grupos).map((k: string) => h('div', { key: k, style: { display: 'flex', flexDirection: 'column', gap: 6 } },
        h('div', { style: { display: 'flex', alignItems: 'center', gap: 4 } }, icone('map-pin', C.suave, 's'), rotulo('Saída: ' + k)),
        h('div', { style: { display: 'flex', flexWrap: 'wrap', gap: 4 } },
          ...grupos[k].map((r: any, i: number) => {
            const passou = diaUtil && r._m < minAgora
            const ehProx = proxima && r === proxima
            return h('span', { key: i, style: { padding: '3px 8px', borderRadius: 4, fontSize: 13, fontVariantNumeric: 'tabular-nums',
              border: `1px solid ${ehProx ? C.ouro : C.borda}`, background: ehProx ? C.ouro : C.fundo,
              color: passou ? C.apagado : C.texto, fontWeight: ehProx ? 700 : 500 } }, r._h)
          })))),
      obs.length ? h('div', null, rotulo('Observação'), h('div', { style: { fontSize: 12, color: C.texto2 } }, obs.join(' · '))) : null,
      h('div', { style: { fontSize: 11, color: C.apagado, marginTop: 'auto' } }, `${hor.length} horários · fonte: setor de transportes`)
    ])
  } catch (e: any) {
    console.error('[Horários da Linha]', e)
    return caixa([aviso('danger', 'exclamation-mark-triangle', null, 'Erro no widget: ' + String(e?.message || e))])
  }
}
''',
    'src/runtime/translations/default.ts': r'''export default {
  _widgetLabel: 'Horários da Linha'
}
''',
  },
  'servicos-passageiro': {
    'manifest.json': r'''{
  "name": "servicos-passageiro",
  "label": "Serviços ao Passageiro",
  "type": "widget",
  "version": "1.19.0",
  "exbVersion": "1.19.0",
  "author": "GeoAPUC",
  "description": "Informações e atalhos úteis ao passageiro (integração, app de horários, tarifas e recarga) no padrão visual dos widgets GeoAPUC.",
  "copyright": "",
  "license": "",
  "properties": {},
  "translatedLocales": ["en"],
  "defaultSize": { "width": 380, "height": 320 }
}
''',
    'config.json': r'''{
  "urlGooglePlay": "https://play.google.com/store/apps/details?id=com.apucarana.voudeonibus",
  "urlAppStore": "https://apps.apple.com/br/app/vou-de-%C3%B4nibus-apucarana/id1488768260",
  "urlTarifas": "http://www.valap.com.br/tarifas.html",
  "urlLojas": "http://www.valap.com.br/lojas/9/apucarana/48/centro.html"
}
''',
    'icon.svg': r'''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 16 16" width="16" height="16"><rect x="3" y="1.5" width="10" height="11" rx="2" fill="none" stroke="#000" stroke-width="1.2"/><rect x="4.6" y="3.2" width="6.8" height="3.6" rx=".6" fill="#000"/><circle cx="5.5" cy="9.6" r=".9" fill="#000"/><circle cx="10.5" cy="9.6" r=".9" fill="#000"/><rect x="4" y="12.5" width="2" height="2" rx=".5" fill="#000"/><rect x="10" y="12.5" width="2" height="2" rx=".5" fill="#000"/></svg>
''',
    'src/runtime/widget.tsx': r'''// Serviços ao Passageiro – GeoAPUC (v1: padrão visual dos widgets dos DEVs — Calcite)
// Escrito sem JSX (React.createElement) para não depender da configuração de compilação.
import { React, type AllWidgetProps } from 'jimu-core'
import { CalciteButton, CalciteIcon } from 'calcite-components'

const h = React.createElement
const C = { fundo: '#FFFFFF', borda: '#E5E7EB', texto: '#1F2937', texto2: '#4B5563', marca: '#4463A9' }
const FONTE = "Inter, 'Avenir Next', Roboto, Arial, sans-serif"

export default function Widget (props: AllWidgetProps<any>) {
  const cfg: any = props.config || {}
  const botao = (txt: string, url: string, ic: string, fimExterno = true) => h(CalciteButton as any, {
    href: url, target: '_blank', rel: 'noopener noreferrer', appearance: 'outline', kind: 'brand', scale: 's', width: 'full',
    iconStart: ic, iconEnd: fimExterno ? 'launch' : undefined
  }, txt)
  const secao = (ic: string, titulo: string, texto: any, botoes: any[] = [], ultima = false) => h('div', {
    style: { display: 'flex', gap: 10, padding: '10px 0', borderBottom: ultima ? 'none' : `1px solid ${C.borda}` }
  },
    h(CalciteIcon as any, { icon: ic, scale: 'm', style: { color: C.marca, flex: 'none', marginTop: 1 } }),
    h('div', { style: { display: 'flex', flexDirection: 'column', gap: 6, minWidth: 0, flex: 1 } },
      h('div', { style: { fontSize: 13, fontWeight: 600, color: C.texto } }, titulo),
      h('div', { style: { fontSize: 12, color: C.texto2, lineHeight: 1.4 } }, texto),
      botoes.length ? h('div', { style: { display: 'grid', gridTemplateColumns: `repeat(${botoes.length}, 1fr)`, gap: 6 } }, ...botoes) : null))

  return h('div', {
    className: 'jimu-widget servicos-passageiro',
    style: { width: '100%', height: '100%', boxSizing: 'border-box', background: C.fundo, borderRadius: 8, border: `1px solid ${C.borda}`,
             padding: '12px 14px', overflowY: 'auto', fontFamily: FONTE, color: C.texto }
  },
    h('div', { style: { fontSize: 15, fontWeight: 600, color: C.texto, marginBottom: 2 } }, 'Serviços ao passageiro'),
    secao('route-to', 'Apucarana Integrada',
      'Pague uma passagem e troque de linha em qualquer ponto da cidade, sem passar pelo Terminal, usando o Cartão Passe Fácil ou o app BiPay.'),
    secao('mobile', 'App Vou de Ônibus Apucarana', 'Horários, mapa da linha e previsão de chegada no celular.',
      [botao('Google Play', cfg.urlGooglePlay, 'download', false), botao('App Store', cfg.urlAppStore, 'download', false)]),
    secao('credit-card', 'Tarifas, cartão e recarga', 'A 1ª via do Passe Fácil é gratuita na VAL, ao lado do Terminal Urbano.',
      [botao('Tarifas', cfg.urlTarifas, 'tag'), botao('Lojas e recarga', cfg.urlLojas, 'map-pin')], true))
}
''',
    'src/runtime/translations/default.ts': r'''export default {
  _widgetLabel: 'Serviços ao Passageiro'
}
''',
  },
}

for nome, arquivos in WIDGETS.items():
    destino = os.path.join(WIDGETS_DIR, nome)
    if os.path.isdir(destino):
        shutil.copytree(destino, os.path.join(PASTA_BKP, "widget_" + nome), dirs_exist_ok=True)
    for rel, conteudo in arquivos.items():
        p = os.path.join(destino, *rel.split("/"))
        os.makedirs(os.path.dirname(p), exist_ok=True)
        with open(p, "w", encoding="utf-8", newline="\n") as f:
            f.write(conteudo)
    log(f"  · {nome}: {len(arquivos)} arquivos gravados")
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Config do app 3 =====
log("\n▶ CÉLULA 3 — Ajustes no config do app 3: iniciando...")
import base64
shutil.copy2(CFG_APP, os.path.join(PASTA_BKP, "app3_config_antes.json"))
with open(CFG_APP, encoding="utf-8") as f:
    cfg = json.load(f)
W, L = cfg["widgets"], cfg["layouts"]
MARCA, TEXTO, SUAVE, BORDA = "#4463A9", "#1F2937", "#6B7280", "#E5E7EB"
FONTE = "Inter, 'Avenir Next', Roboto, Arial, sans-serif"
SVG_BUS = r"""<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path fill="#6B7280" d="M22.025 7.01V5.835a.85.85 0 0 0-.847-.848h-1.065c-.615-1.932-1.345-2.383-1.468-2.445C18.489 2.452 17.448 2 12.25 2c-5.55 0-6.734.467-6.871.53-.137.069-.874.523-1.492 2.457H2.822a.85.85 0 0 0-.847.848V7.01A1.02 1.02 0 0 0 1 8.025v2.11c0 .932 1.185 1.55 1.969 1.55.016 0 .03-.008.047-.009-.007.332-.016 7.249-.016 7.249 0 .519.012 1.811 1 2.03v.85C4 22.465 4.536 23 5.194 23h1.612C7.464 23 8 22.464 8 21.806V21h1v1h6v-1h1v.806c0 .658.536 1.194 1.194 1.194h1.612c.658 0 1.194-.536 1.194-1.194v-.85c.988-.22 1-1.512 1-2.031 0 0-.002-5.55 0-5.551 0 0-.009-1.366-.016-1.698.017 0 .03.01.047.01.784 0 1.969-.619 1.969-1.55v-2.11c0-.548-.434-.988-.975-1.016M3 10.744c-.453-.007-1-.385-1-.573V8h1zM3.398 7.1a1 1 0 0 0-.423-.1V5.987h.645q-.12.507-.222 1.114zM5.8 3.336c.049-.019 1.187-.436 6.451-.436 4.913 0 5.88.404 5.895.41.006.003.203.242.464.69H5.377c.247-.432.427-.664.423-.664M4.934 5H19.06c.46 1.287.922 3.566.936 7.511-2.219.295-5.298.486-7.996.489-2.7-.003-5.78-.194-7.998-.489.014-3.95.477-6.228.93-7.511zM4 16.994c.46.02 1.091.079 1.665.24.186.052.248.146.429.506l.136.309c-1.163-.045-1.852-.183-2.23-.29zm3 4.812a.194.194 0 0 1-.194.194H5.194A.194.194 0 0 1 5 21.806V21h2zM14 21h-4v-1h4zm5 .806a.194.194 0 0 1-.194.194h-1.612a.194.194 0 0 1-.194-.194V21h2zm1-4.049c-.382.11-1.07.247-2.23.292l.126-.287c.19-.382.253-.476.439-.528.574-.16 1.205-.219 1.665-.24zm0-1.762c-.518.021-1.25.085-1.935.276-.63.176-.86.637-1.073 1.065l-.375.849a.6.6 0 0 0 .023.59.61.61 0 0 0 .518.288c1.36-.01 2.259-.135 2.842-.27v.132C20 20 19.804 20 19.71 20H15v-1H9v1H4.29C4.196 20 4 20 4 18.925v-.132c.583.135 1.483.26 2.846.27.208 0 .405-.11.508-.278a.59.59 0 0 0 .041-.574l-.397-.897c-.202-.406-.433-.867-1.063-1.043-.685-.191-1.417-.255-1.935-.276v-1.018c1.167.152 2.546.28 4.046.367C8.439 16.968 9.858 17 10.039 17h3.922c1.05 0 1.764-.717 1.993-1.656A57 57 0 0 0 20 14.976v1.02zm0-2.023c-1.643.219-3.255.33-4.837.414-.182.726-.256 1.614-1.202 1.614h-3.922a1.02 1.02 0 0 1-.992-.774l-.21-.84C7.234 14.3 5.647 14.19 4 13.972v-.454c2.252.29 5.314.48 8.001.482 2.685-.002 5.747-.191 7.999-.482zm.602-6.872a18 18 0 0 0-.22-1.113h.643v1.014a1 1 0 0 0-.423.1zM22 10.171c0 .188-.547.566-1 .573V8h1z"/></svg>"""
BUS_URI = "data:image/svg+xml;base64," + base64.b64encode(SVG_BUS.encode()).decode()

def resto(a, b):
    if str(a).endswith("%") and str(b).endswith("%"):
        return f"{round(100 - float(a[:-1]) - float(b[:-1]), 2)}%"
    return ""
def item(wid, left, top, width, height):
    return {"type": "WIDGET", "widgetId": wid, "isPending": False,
            "bbox": {"left": left, "right": resto(left, width), "top": top, "bottom": resto(top, height), "width": width, "height": height},
            "setting": {"autoProps": {"left": True, "right": False, "top": True, "bottom": False}, "vCenter": False, "hCenter": False}}

# 3.1 — sai: título + 3 cartões de texto + 4 botões pílula dos serviços
SAEM = ["widget_tit_serv", "widget_card_integra", "widget_card_app", "widget_card_tarifa",
        "widget_btn_gplay", "widget_btn_appstore", "widget_btn_tarifas", "widget_btn_lojas"]
for lay in L.values():
    for k in [k for k, it in lay.get("content", {}).items() if it.get("widgetId") in SAEM]:
        del lay["content"][k]
        if k in lay.get("order", []): lay["order"].remove(k)
for w in SAEM:
    if W.pop(w, None): log("  · removido:", w)

# 3.2 — entra: widget Serviços ao Passageiro
W["widget_servicos"] = {"uri": "widgets/servicos-passageiro/", "version": "1.19.0", "id": "widget_servicos",
                        "label": "Serviços ao Passageiro", "config": json.loads(WIDGETS["servicos-passageiro"]["config.json"])}
p0 = L["layout_p0"]
for k in [k for k, it in p0["content"].items() if it.get("widgetId") in ("widget_servicos", "widget_horarios")]:
    del p0["content"][k]
    if k in p0.get("order", []): p0["order"].remove(k)
for novo in (item("widget_horarios", "74%", "1%", "25%", "60%"), item("widget_servicos", "74%", "62%", "25%", "37%")):
    k = str(max([int(x) for x in p0["content"]] + [0]) + 1)
    p0["content"][k] = novo; p0.setdefault("order", []).append(k)
log("  · Serviços ao Passageiro na coluna direita, abaixo da Tabela de horários")

# 3.3 — Horários da Linha: config nova (mantém os endereços de paradas e Mobilibus)
hc = W["widget_horarios"]["config"]
novo = json.loads(WIDGETS["horarios-linha"]["config.json"])
for k in ("paradasUrl", "paradasLayerId", "mobilibusUrl", "mobilibusLinhas", "linhasUrl", "horariosUrl", "dataSourceId"):
    if hc.get(k): novo[k] = hc[k]
W["widget_horarios"]["config"] = novo
log("  · Horários da Linha → visual Calcite (painel branco, ouro #F5C400 na próxima saída)")

# 3.4 — Lista: ícone Calcite, título à esquerda, seleção discreta
W["widget_card_icone"]["config"]["functionConfig"]["imageParam"].update({"url": BUS_URI, "originalUrl": BUS_URI})
t = W["widget_tit_lista"]["config"]
t["text"] = (f'<p style="margin:0"><span style="font-size:15px;font-weight:600;color:{TEXTO};font-family:{FONTE}">Linhas de ônibus</span></p>'
             f'<p style="margin:2px 0 0 0"><span style="font-size:12px;color:{SUAVE};font-family:{FONTE}">Escolha uma linha para ver o trajeto e os horários</span></p>')
t["style"]["verticalAlign"] = "center"
def bst(bg=None, sel=False):
    s = {"border": {"border": {"type": "solid", "color": "", "width": "0px"}}, "borderRadius": {"number": [4, 4, 4, 4], "unit": "px"}}
    if bg: s["background"] = {"color": bg, "fillType": "fill", "image": {"url": ""}}
    if sel: s["border"] = {"borderLeft": {"type": "solid", "color": MARCA, "width": "3px"}}
    return s
cc = W["widget_lista"]["config"]["cardConfigs"]
cc["DEFAULT"]["backgroundStyle"] = bst()
cc["HOVER"]["backgroundStyle"] = bst("#F5F7FA")
cc["SELECTED"]["backgroundStyle"] = bst("#EFF6FF", True)
W["widget_lista"]["style"] = {"background": {"color": "#FFFFFF", "fillType": "fill", "image": {"url": ""}},
                             "borderRadius": {"number": [8, 8, 8, 8], "unit": "px"}, "border": {"type": "solid", "color": BORDA, "width": "1px"}}
# texto do cartão: número na cor da marca + nome; 2ª linha cinza
ct = W["widget_card_texto"]["config"]
ct["text"] = (ct["text"]
              .replace("font-size:14px;font-weight:700;color:#1b2a4a", f"font-size:14px;font-weight:600;color:{TEXTO};font-family:{FONTE}")
              .replace("font-size:12px;color:#6b7690", f"font-size:12px;color:{SUAVE};font-family:{FONTE}"))
log("  · Lista: ícone de ônibus do Calcite (cinza, sem círculo), título à esquerda, seleção com faixa azul fina")

cfg["timestamp"] = int(datetime.datetime.now().timestamp() * 1000)
with open(CFG_APP, "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)
log("  config.json escrito:", CFG_APP, "| widgets:", len(W))
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — Próximos passos =====
log("\n▶ CÉLULA 4 — Próximos passos: iniciando...")
log("  1) Reinicie o ExB (widget novo): feche as janelas do ExB e dê dois cliques no Iniciar_ExB.bat; espere o client compilar")
log("  2) Construtor: Ctrl+F5 → mudança pequena → Salvar → Publicar")
log("  3) Teste: lista (ícone cinza), Tabela de horários branca com botão 'em tempo real', interruptor Paradas e Serviços ao passageiro")
log("   Construtor (app 3): https://localhost:3001/builder/?id=3")
log("   Rascunho:           https://localhost:3001/experience/3/?draft=true&locale=pt-br")
log("   Prévia local:       https://localhost:3001/experience/3/?locale=pt-br")
log(f"  Backups em: {PASTA_BKP}")
log(f"  Saída completa: {ARQ_TXT}")
log("  Ao terminar: feche esta aba e remova o notebook do projeto (Remove, não Delete).")
log("✔ CÉLULA 4 concluída com sucesso.")
